# 01 — Render sanity check

Does the sim camera render, and does a real webcam capture come back in the
same shape? This is the Phase 1 done-when, made visual.

Outputs are intentionally cleared in version control. Run top-to-bottom; the
webcam cell degrades gracefully if no camera is available.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from sensorforge.capture.webcam import Webcam
from sensorforge.sim.camera import SimCamera
from sensorforge.sim.renderer import SimRenderer

SCENE = Path.cwd().parent / "scenes" / "checkerboard.xml"

## Sim render

The renderer returns float32 RGB in [0, 1], treated as approximately-linear
intensity (see `sim/renderer.py` for the linearity caveat).

In [ ]:
cam = SimCamera.from_scene(SCENE)
print("intrinsics:", cam.intrinsics.model_dump())
print("fx_px:", round(cam.intrinsics.fx_px, 1), "fovx_deg:", round(cam.intrinsics.fovx_deg, 1))

with SimRenderer(cam) as r:
    sim = r.render()
print("sim frame:", sim.shape, sim.dtype, "range", (float(sim.min()), float(sim.max())))

## Real capture

Point the webcam at a printed checkerboard. On macOS the first run triggers a
camera-permission prompt. If no camera is present we fall back to showing the
sim render alone.

In [ ]:
real = None
try:
    with Webcam(index=0) as wc:
        wc.lock_exposure_and_white_balance()
        real = wc.grab()
    print("real frame:", real.shape, real.dtype)
except RuntimeError as e:
    print("no webcam available, sim-only:", e)

## Side by side

Shapes should match (H, W, 3). The intensities will *not* match yet: that gap
(linear sim vs gamma-encoded sRGB webcam, plus the missing ISP) is exactly
what Phases 2-4 close.

In [ ]:
panels = [("sim render (linear)", sim)]
if real is not None:
    panels.append(("real capture (sRGB)", real))
    assert sim.shape == real.shape, (sim.shape, real.shape)

fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4))
axes = np.atleast_1d(axes)
for ax, (title, img) in zip(axes, panels, strict=False):
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()